# Loan Default Prediction — Phase 2 EDA

Inspect raw data quality and relationships. The notebook never modifies the source CSV or removes valid outliers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'loan-default-prediction']
ROOT = next(
    (candidate for candidate in CANDIDATE_ROOTS if (candidate / 'data/raw/Loan_default.csv').is_file()),
    Path.cwd(),
)
DATA_PATH = ROOT / 'data/raw/Loan_default.csv'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Raw dataset not found: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
NUMERICAL = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Default']
CATEGORICAL = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.dtypes.to_frame('dtype'))
display(df.head())
display(df.sample(5, random_state=42))
display(df[NUMERICAL].describe().T)
display(df[CATEGORICAL].describe().T)

## Missing values, duplicates, and value ranges

In [ ]:
missing = pd.DataFrame({'Column': df.columns, 'Missing Count': df.isna().sum().values, 'Missing %': (df.isna().mean() * 100).values})
display(missing)
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Rows with duplicate LoanID:', int(df['LoanID'].duplicated(keep=False).sum()))
if df.isna().sum().sum() == 0:
    print('No missing-value imputation is required for the current dataset.')
checks = {'Age <= 0': df.Age <= 0, 'Age < 18': df.Age < 18, 'Age > 100': df.Age > 100, 'Income <= 0': df.Income <= 0, 'LoanAmount <= 0': df.LoanAmount <= 0, 'CreditScore < 300': df.CreditScore < 300, 'CreditScore > 850': df.CreditScore > 850, 'MonthsEmployed < 0': df.MonthsEmployed < 0, 'NumCreditLines < 0': df.NumCreditLines < 0, 'InterestRate < 0': df.InterestRate < 0, 'LoanTerm <= 0': df.LoanTerm <= 0, 'DTIRatio < 0': df.DTIRatio < 0}
display(pd.Series({name: int(mask.sum()) for name, mask in checks.items()}, name='Suspicious count').to_frame())
iqr_rows = []
for column in NUMERICAL[:-1]:
    q1, q3 = df[column].quantile([.25, .75]); iqr = q3 - q1
    iqr_rows.append({'Feature': column, 'p01': df[column].quantile(.01), 'p50': df[column].median(), 'p99': df[column].quantile(.99), 'IQR outlier count': int(((df[column] < q1 - 1.5*iqr) | (df[column] > q3 + 1.5*iqr)).sum())})
display(pd.DataFrame(iqr_rows).set_index('Feature'))
print('IQR outliers are statistical flags only; valid extreme observations are retained.')

In [ ]:
for column in CATEGORICAL:
    values = df[column].astype('string')
    print(f'\n{column}: leading/trailing-space values = {int((values != values.str.strip()).fillna(False).sum())}')
    display(pd.DataFrame({'count': df[column].value_counts(dropna=False), 'percent': df[column].value_counts(dropna=False, normalize=True) * 100}))
    distinct = values.dropna().unique()
    mixed_case = [value for value in distinct if value != value.title()]
    if mixed_case: print('Capitalization values to review (preserved):', mixed_case)

## Target distribution and class imbalance

In [ ]:
counts = df.Default.value_counts().sort_index()
default_rate = counts.get(1, 0) / len(df)
non_default_rate = counts.get(0, 0) / len(df)
print(f'Default = 0: {counts.get(0, 0):,} ({non_default_rate:.2%})')
print(f'Default = 1: {counts.get(1, 0):,} ({default_rate:.2%})')
print(f'Imbalance ratio (majority/minority): {counts.max() / counts.min():.2f}:1')
sns.countplot(data=df, x='Default', color='steelblue')
plt.title('Loan default class distribution'); plt.xlabel('Default (0 = No, 1 = Yes)'); plt.ylabel('Applications'); plt.tight_layout(); plt.show()

## Distributions and default rates

In [ ]:
for column in ['Age', 'Income', 'LoanAmount', 'CreditScore', 'InterestRate', 'DTIRatio']:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(data=df, x=column, bins=40, ax=ax)
    ax.set(title=f'{column} distribution', xlabel=column, ylabel='Count')
    plt.tight_layout(); plt.show()

for column in ['Education', 'EmploymentType', 'LoanPurpose', 'HasMortgage', 'HasDependents']:
    rates = df.groupby(column, dropna=False).Default.agg(['mean', 'count']).sort_values('mean', ascending=False)
    display(rates.rename(columns={'mean': 'Default rate', 'count': 'Applications'}))
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=df, x=column, y='Default', errorbar=None, ax=ax)
    ax.set(title=f'Default rate by {column}', xlabel=column, ylabel='Default rate')
    ax.tick_params(axis='x', rotation=25); plt.tight_layout(); plt.show()

## Numerical correlations
Correlations are descriptive only; no feature is removed based on correlation in Phase 2.

In [ ]:
correlations = df[NUMERICAL].corr(numeric_only=True)
display(correlations)
plt.figure(figsize=(11, 8))
sns.heatmap(correlations, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Correlation heatmap — numeric variables'); plt.tight_layout(); plt.show()